In [1]:
import os
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset, concatenate_datasets

# 1. Charger le dataset Emotion de HuggingFace

# 2. Préparer les textes et les labels
dataset = load_dataset("amineouaki/emotion_englishv3")

texts = dataset['train']['Text'] + dataset['validation']['Text'] + dataset['test']['Text']
labels = dataset['train']['Emotion'] + dataset['validation']['Emotion'] + dataset['test']['Emotion']


# 3. Encodage des labels (facultatif car ils sont déjà des entiers)
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

# 4. Tokenisation des textes
print("Tokenisation des textes...")
max_words = 10000  # taille du vocabulaire
maxlen = 50        # longueur maximale des séquences (SEQUENCE_LENGTH)

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')

# 5. Séparer en train / validation
X_train, X_val, y_train, y_val = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

# 6. Sauvegarder les datasets au format .npy
print("Sauvegarde des jeux de données...")
os.makedirs("./prepared_data", exist_ok=True)

np.save("./prepared_data/X_train.npy", X_train)
np.save("./prepared_data/y_train.npy", y_train)
np.save("./prepared_data/X_val.npy", X_val)
np.save("./prepared_data/y_val.npy", y_val)

print("Fini ! Les fichiers sont prêts pour l'entraînement :")
print("- ./prepared_data/X_train.npy")
print("- ./prepared_data/y_train.npy")
print("- ./prepared_data/X_val.npy")
print("- ./prepared_data/y_val.npy")

# 7. Informations utiles pour configurer gru_svm_main.py


2025-04-30 03:46:22.635687: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745984782.661617   13746 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745984782.671024   13746 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745984782.692744   13746 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745984782.692763   13746 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1745984782.692766   13746 computation_placer.cc:177] computation placer alr

Tokenisation des textes...
Sauvegarde des jeux de données...
Fini ! Les fichiers sont prêts pour l'entraînement :
- ./prepared_data/X_train.npy
- ./prepared_data/y_train.npy
- ./prepared_data/X_val.npy
- ./prepared_data/y_val.npy


In [1]:
import numpy as np
from collections import Counter

# Charger les labels
y_train = np.load('./prepared_data/y_train.npy')
y_val = np.load('./prepared_data/y_val.npy')

# Combiner les deux
y_all = np.concatenate((y_train, y_val))

# Compter les classes
all_counts = Counter(y_all)

# Afficher la distribution
print("Distribution des classes dans l'ensemble combiné (train + val) :")
for cls, count in all_counts.items():
    print(f"Classe {cls} : {count} exemples")


Distribution des classes dans l'ensemble combiné (train + val) :
Classe 1 : 4236 exemples
Classe 6 : 8629 exemples
Classe 3 : 1947 exemples
Classe 2 : 1853 exemples
Classe 0 : 1935 exemples
Classe 4 : 2030 exemples
Classe 5 : 263 exemples


In [2]:
import numpy as np
from collections import Counter

# Dictionnaire de mappage des labels
label_map = {
    0: 'joy',
    1: 'sadness',
    2: 'anger',
    3: 'fear',
    4: 'disgust',
    5: 'surprise'
}

# Charger les labels
y_train = np.load('./prepared_data/y_train.npy')
y_val = np.load('./prepared_data/y_val.npy')

# Combiner les deux
y_all = np.concatenate((y_train, y_val))

# Compter les classes
all_counts = Counter(y_all)

# Afficher la distribution avec les noms d’émotions
print("Distribution des classes dans l'ensemble combiné (train + val) :")
for cls_id, count in all_counts.items():
    emotion = label_map.get(cls_id, f"Unknown({cls_id})")
    print(f"{emotion} ({cls_id}) : {count} exemples")


Distribution des classes dans l'ensemble combiné (train + val) :
sadness (1) : 4236 exemples
Unknown(6) (6) : 8629 exemples
fear (3) : 1947 exemples
anger (2) : 1853 exemples
joy (0) : 1935 exemples
disgust (4) : 2030 exemples
surprise (5) : 263 exemples


In [1]:
!pip freeze > requirements.txt